# ECML26 — Pipeline (neu aufgebaut, repo-losgelöst)

Spiegelt die Paper-Soll-Struktur **Block 1 → 2 → 3 → 4 → 5 → 6**, läuft auf dem
Stage-8-Parallel/SQLite-Motor und enthält die vier Neuerungen:

1. **tie-safe kNN-Purity** (`mode="prop"`) überall, wo Purity Strukturmetrik ist
2. **repeated-OOF-Correctness** je Kunde (`correct_rate_i`), verknüpft mit `purity_i`/`margin_i`
3. **korrigierte Modelle + Label-Arm** als dritter Modelltyp
4. **Parallelisierung + Fold-Cache** (SQLite-WAL, resume-fähig) als Motor

Alles orchestriert `ecml.*` — kein Repo-Layout-Bezug. `USE_SYNTHETIC=True` fährt end-to-end
ohne Echtdaten. Ausführen im **numpy<2-Env mit `proto_dist_ml==1.0.1`**.

In [2]:
# --- Setup -------------------------------------------------------------------
import sys
from pathlib import Path

PKG_PARENT = Path.cwd()          # Ordner, der 'ecml/' enthält; ggf. anpassen
if str(PKG_PARENT) not in sys.path:
    sys.path.insert(0, str(PKG_PARENT))



import numpy as np, pandas as pd
from src import data, search, repeated_oof, engine, weight_profiles as WP
from src.single_view_mglvq.mglvq_fast import MGLVQ
from src.structural.run import run_structural_analysis
from src.single_view_mglvq.run import run_single_view_mglvq
from src.analysis.run import run_pointwise_analysis
from src.analysis import paper_tables as PT
from src.analysis.figures import plot_structure_vs_prediction_boxplot
print("ecml importiert.")

ecml importiert.


In [3]:
import os

print(os.getcwd())

/home/jovyan/work/workspace/1.0_project_repos/ECML26_new


In [4]:
# --- Konfiguration -----------------------------------------------------------
USE_SYNTHETIC = False
PRODUCT       = "0201"

REPO_DATA_DIR = None             # Path zu data/raw/... (Repo-Layout) — sonst None
FLAT = dict(                     # flache Uploads (nur bei USE_SYNTHETIC=False & REPO_DATA_DIR=None)
    matrix_paths=dict(
        naics="/home/jovyan/work/workspace/1.0_project_repos/ECML26/data/raw/D/ECML/0201/dissimilarity_naics_matrix.pkl",
        hs="/home/jovyan/work/workspace/1.0_project_repos/ECML26/data/raw/D/ECML/0201/dissimilarity_hs_matrix.pkl",
        am="/home/jovyan/work/workspace/1.0_project_repos/ECML26/data/raw/D/ECML/0201/dissimilarity_am_A_matrix.pkl",
    ),
    target_path="/home/jovyan/work/workspace/1.0_project_repos/ECML26/data/raw/y/ECML/0201/A.pkl",
    target_col="A",
)

OUT      = Path("outputs"); OUT.mkdir(exist_ok=True)
STRUCT   = OUT / "structural"
SV       = OUT / "single_view_mglvq"
ANALYSIS = OUT / "analysis"
CACHE_DB = OUT / f"cache_{PRODUCT}.sqlite"

N_JOBS = 2; BASE_SEED = 42; RANDOM_STATE = 42
if USE_SYNTHETIC:
    SEARCH = dict(T=60, n_splits=3);  ROOF = dict(T=60, n_splits=5, n_repeats=4)
else:
    SEARCH = dict(T=150, n_splits=3); ROOF = dict(T=150, n_splits=10, n_repeats=5)
print("USE_SYNTHETIC =", USE_SYNTHETIC)

USE_SYNTHETIC = False


In [5]:
# --- Daten laden (einmal aligned; alle Blöcke teilen die Kundenordnung) -------
if USE_SYNTHETIC:
    bundle = data.synthetic_bundle(m=120, seed=0)
elif REPO_DATA_DIR is not None:
    bundle = data.load_bundle(base_path=REPO_DATA_DIR, product=PRODUCT)
else:
    bundle = data.load_bundle(product=PRODUCT, **FLAT)

print("m =", len(bundle.y), "| Klassen:", dict(zip(*np.unique(bundle.y, return_counts=True))))
print("Views:", list(bundle.matrices_raw.keys()))

m = 2025 | Klassen: {0: 987, 1: 1038}
Views: ['naics', 'hs', 'am']


## Block 1 — Strukturanalyse (tie-safe Purity)
Single-view Summary + `pointwise_structure_df` (purity_i/margin_i je View, **tie-safe** `prop`),
Cross-View Korrelation/kNN-Overlap/Agreement.

In [6]:
struct = run_structural_analysis(
    matrices=bundle.matrices_raw, y=bundle.y_series,
    run_name=PRODUCT, out_root=STRUCT, k=10,
    n_perm=(10 if USE_SYNTHETIC else 50),
)
struct_dir = struct["run_dir"]
struct["single_summary_df"]

[WARN] Few valid points for 2NN intrinsic dimension: 15 / 2025


,matrix,n,sym_max_abs_diff,diag_min,diag_max,D_min,D_max,D_mean_all,D_std_all,dist_mean_ut,...,knn_purity_mean,n_class0,n_class1,knn_purity_baseline_mean,knn_purity_baseline_std,knn_purity_z,robust_mu_mean,robust_mu_std,robust_mu_frac_neg,robust_mu_frac_zero
0,naics,2025,0.0,0.0,0.0,0.0,1.0,0.852188,0.208230,0.852609,...,0.542185,987,1038,0.500355,0.002661,15.717320,-0.014774,0.135176,0.080000,0.879506
1,hs,2025,0.0,0.0,0.0,0.0,1.0,0.913360,0.147304,0.913811,...,0.522793,987,1038,0.500044,0.003073,7.402061,-0.007715,0.116938,0.320988,0.429136
2,am,2025,0.0,0.0,0.0,0.0,1.0,0.690013,0.199430,0.690354,...,0.519249,987,1038,0.500134,0.002187,8.741289,-0.000771,0.027603,0.156049,0.693333


## Block 2 — Single-View-MGLVQ (OOF-Baselines)
Grid über K, out-of-fold Predictions je View; verknüpft Struktur→Prädiktion punktweise.

In [7]:
sv = run_single_view_mglvq(
    matrices=bundle.matrices_raw, y=bundle.y_series, model_cls=MGLVQ,
    run_name=PRODUCT, out_root=SV,
    n_splits=(3 if USE_SYNTHETIC else 5),
    param_grid={"K": [3, 5, 7]} if USE_SYNTHETIC else {"K": [3, 4, 5, 6, 7, 8, 9, 10]},
    pointwise_structure_df=struct["pointwise_structure_df"],
)
sv_dir = sv["run_dir"]
sv["summary_df"]

Matrizen 0201:   0%|          | 0/3 [00:00<?, ?it/s]

,matrix,best_score_balanced_accuracy,balanced_accuracy_from_oof,recall_from_oof,precision_from_oof,specificity_from_oof,f1_from_oof,tp,fp,tn,fn,best_K
0,naics,0.586540,0.563776,0.492293,0.586682,0.635258,0.535359,511,360,627,527,9
1,hs,0.577714,0.573379,0.610790,0.580586,0.535968,0.595305,634,458,529,404,4
2,am,0.556348,0.553407,0.638728,0.558081,0.468085,0.595687,663,525,462,375,3


### Pointwise-Merge (Struktur ⊕ Single-View)
`wide_df` mit `knn_purity_{naics,hs,am}` (tie-safe) und `correct_{...}` — Basis für Tab. 7/9 + Figur.

In [8]:
pw = run_pointwise_analysis(
    structural_run_dir=struct_dir, single_view_run_dir=sv_dir,
    out_dir=ANALYSIS, structure_metric="knn_purity",
)
wide_df = pw["wide_df"]
wide_df.head()

,customer_id,y_am,y_hs,y_naics,mu_am,mu_hs,mu_naics,d_plus_am,d_plus_hs,d_plus_naics,...,y_true_hs,y_true_naics,y_pred_am,y_pred_hs,y_pred_naics,correct_am,correct_hs,correct_naics,y_ref,y_true_ref
0,1,1,1,1,0.000000,0.000000,0.0,0.000000,0.500000,0.00,...,1,1,1,0,0,1,0,0,1,1
1,10,1,1,1,0.000000,-0.012245,0.0,0.000000,0.504167,0.15,...,1,1,1,1,0,1,1,0,1,1
2,100,0,0,0,-0.043478,-0.029412,-1.0,0.343750,0.654762,0.00,...,0,0,0,1,0,1,0,1,0,0
3,1000,0,0,0,0.045455,-0.076923,0.0,0.479167,0.500000,0.00,...,0,0,0,0,1,1,1,0,0,0
4,1001,0,0,0,0.000000,0.000000,0.0,0.479167,0.275000,0.00,...,0,0,0,1,0,1,0,1,0,0


## Block 3 — Multi-View-Suche (Engine: bestes (K, η) je Modelltyp)
Paralleler Grid über Global + Label auf dem Fold-Cache. `staged_search` (coarse→refine→finalists)
für den Echtlauf, `full_search` mit Mini-Grid für die Synthetik.

In [9]:
engine.init_cache_db(CACHE_DB)

# Paper-Grid (original ECML, erschöpfend)
paper_k_pairs = [(k0, k1) for k0 in range(3, 11) for k1 in range(3, 11)]  # K0,K1 ∈ 3..10

paper_etas    = [0.003, 0.0075, 0.01, 0.015, 0.025, 0.03, 0.04, 0.05]
paper_etas    = [0.003, 0.0075, 0.01, 0.015, 0.025, 0.03, 0.04, 0.05]
paper_etas    = [0.03, 0.04, 0.05]



paper_single_k = [3, 4, 5, 6, 7, 8, 9, 10]

if USE_SYNTHETIC:
    results, best = search.full_search(
        bundle.DL, bundle.y, CACHE_DB,
        T=SEARCH["T"], n_splits=SEARCH["n_splits"],
        random_state=RANDOM_STATE, base_seed=BASE_SEED, n_jobs=N_JOBS,
        single_k=[3, 5], k_pairs=[(3, 3), (5, 5)], etas=[0.02, 0.04],
    )
else:
    # 1:1 Paper-Reproduktion: erschöpfendes Grid, 10-fold
    results, best = search.full_search(
        bundle.DL, bundle.y, CACHE_DB,
        T=150, n_splits=10,
        random_state=RANDOM_STATE, base_seed=BASE_SEED, n_jobs=N_JOBS,
        single_k=paper_single_k,
        k_pairs=paper_k_pairs,
        etas=paper_etas,
    )

print("Beste Configs je Typ:")
for m_, c_ in best.items():
    print(" ", m_, c_)
results.sort_values("bal_acc", ascending=False).head(8)

full:   1%|1         | 5/408 [00:00<?, ?cfg/s]

/home/jovyan/work/workspace/1.0_project_repos/ECML26_new/src/single_view_mglvq/mglvq_fast.py:227: RuntimeWarning: MGLVQ loss bookkeeping drift: expected -251.475, got -231.634 (using recomputed value).
  warnings.warn(
/home/jovyan/work/workspace/1.0_project_repos/ECML26_new/src/single_view_mglvq/mglvq_fast.py:227: RuntimeWarning: MGLVQ loss bookkeeping drift: expected -251.499, got -231.634 (using recomputed value).
  warnings.warn(
/home/jovyan/work/workspace/1.0_project_repos/ECML26_new/src/single_view_mglvq/mglvq_fast.py:227: RuntimeWarning: MGLVQ loss bookkeeping drift: expected -301.369, got -253.353 (using recomputed value).
  warnings.warn(
/home/jovyan/work/workspace/1.0_project_repos/ECML26_new/src/single_view_mglvq/mglvq_fast.py:227: RuntimeWarning: MGLVQ loss bookkeeping drift: expected -303.306, got -238.767 (using recomputed value).
  warnings.warn(
/home/jovyan/work/workspace/1.0_project_repos/ECML26_new/src/single_view_mglvq/mglvq_fast.py:227: RuntimeWarning: MGLVQ loss

KeyboardInterrupt: 

## Block 4 — repeated-OOF-Correctness + Verknüpfung (Neuerung 2)
Je Gewinner-Config R Wiederholungen (reseed der Fold-Layout → nur weitere gecachte Fold-Jobs),
per-Kunde-Prediction via `va_idx` aggregiert → `correct_rate_i`, verknüpft mit `purity_i`/`margin_i`.

In [ ]:
linked, manifest = repeated_oof.run_and_link(
    best, bundle.DL, bundle.y, bundle.codes,
    pointwise_df=wide_df,
    cache_db=CACHE_DB,
    T=ROOF["T"], n_splits=ROOF["n_splits"], n_repeats=ROOF["n_repeats"],
    random_state0=1000, base_seed=BASE_SEED, n_jobs=N_JOBS,
)
for m_, df_ in linked.items():
    print(m_, "| Kunden:", len(df_), "| mittlere correct_rate:",
          round(float(df_["correct_rate"].mean()), 3))
list(linked.values())[0].head()

## Block 3b — Gewichtsprofil-Cluster + Fixed-Weight-Profile
Die per-Fold-Gewichte (a²) der Global-Suche liegen bereits im Cache. Wir clustern sie,
bilden Fixed-Profile (kanonische Single-View-Ecken + Cluster-Zentroide) und evaluieren sie
mit **eingefrorenen** Gewichten (`eta=0`, `v_init=√mixture` wegen der quadratischen Param).
Ergebnis: `fixed_oof_df` je Kunde → speist Tabelle 9.

In [ ]:
fw = WP.collect_fold_weights(CACHE_DB, model="M3GLVQ_Global")
print("Global per-Fold-Gewichte im Cache:", len(fw))

if len(fw) >= 2:
    df_high, thr = WP.select_top_runs(fw, quantile=(0.0 if USE_SYNTHETIC else 0.9))
    clus = WP.cluster_weight_profiles(df_high, n_clusters=(2 if USE_SYNTHETIC else 5))
    try:
        display(WP.cluster_summary(clus))
    except NameError:
        print(WP.cluster_summary(clus))
    g = best.get("M3GLVQ_Global", {"K0": 6, "K1": 6})
    profiles = WP.build_fixed_profiles(clus, K=(g["K0"], g["K1"]))
else:
    profiles = WP.build_fixed_profiles(None, K=(3, 3))   # nur kanonische Ecken

fixed_oof_df = WP.run_fixed_profiles(
    profiles, bundle.DL, bundle.y, bundle.codes, CACHE_DB,
    T=ROOF["T"], n_splits=(3 if USE_SYNTHETIC else 10),
    random_state=RANDOM_STATE, base_seed=BASE_SEED, n_jobs=N_JOBS,
)
profile_map = WP.default_profile_map(profiles)
print("Fixed-Profile:", [p["profile_name"] for p in profiles])
fixed_oof_df.head()

## Block 5 — Paper-Tabellen 5.1–5.9 (tie-safe konsistent)

In [ ]:
tables = {}
tables["distance_distribution"]      = PT.build_distance_distribution_table(bundle.matrices_norm)
tables["structural_characteristics"] = PT.build_structural_characteristics_table(struct["single_summary_df"])
tables["pearson"]                    = PT.build_pearson_correlation_table(struct_dir)
tables["knn_overlap"]                = PT.build_knn_overlap_table(struct_dir, k=10)
tables["agreement"]                  = PT.build_agreement_table(struct_dir)
tables["single_view_baseline"]       = PT.build_single_view_baseline_table(sv["summary_df"])
tables["structure_prediction_dom"]   = PT.build_structure_prediction_dominance_table(
    wide_df, structure_metric="knn_purity")
# Tab. 9 (fixed-profile dominance) braucht die Fixed-Weight-OOF-Suite — optional, separat.

for name, t in tables.items():
    print("\n###", name)
    try:
        display(t)
    except NameError:
        print(t)

In [ ]:
from src.analysis.paper_tables import build_fixed_profile_dominance_table
try:
    t9 = build_fixed_profile_dominance_table(wide_df, fixed_oof_df, profile_map)
    print("### fixed_profile_dominance (Tab. 9)")
    try:
        display(t9)
    except NameError:
        print(t9)
except Exception as e:
    print("Tab. 9 benötigt fixed_oof_df aus Block 3b:", e)

### Sanity-Check (Arbeitsanweisung §2)
Aggregat-Ranking der kNN-Purity muss **Industry > Products > Applications** bleiben
(die *punktweise* Dominanz darf sich durch tie-safe verschieben, das Aggregat nicht).

In [ ]:
chars = tables["structural_characteristics"].set_index("Matrix")["kNN Pur."]
print(chars.sort_values(ascending=False))
print("Ranking:", " > ".join(list(chars.sort_values(ascending=False).index)[:3]))

## Block 6 — Figur: Struktur-vs-Prädiktion (Boxplot über Purity)

In [ ]:
joined_long = Path(ANALYSIS) / "pointwise" / "structure_single_view_joined_long.csv"
try:
    fig = plot_structure_vs_prediction_boxplot(str(joined_long), metric="knn_purity")
    fig
except Exception as e:
    print("Figur benötigt joined_long.csv aus Block-5-Pointwise:", e)

## Phase 6 — Lernkurven-Kopf (alt/neu, implementierst du selbst)
Modelle liegen sauber vor. Global unterscheidet sich bei **jedem** η (Simplex→quadratisch);
Label erst bei η ≳ 0.03 (best-so-far greift bei nicht-monotoner Kurve).

In [ ]:
from src.models import GlobalOld, GlobalNew, LabelOld, LabelNew
# Skelett:
# etas = [0.01, 0.03, 0.05]
# for name, cls in [("Global OLD", GlobalOld), ("Global NEW", GlobalNew),
#                   ("Label OLD", LabelOld),  ("Label NEW", LabelNew)]:
#     for eta in etas:
#         clf = cls(K={0: 6, 1: 6}, T=SEARCH["T"], eta=eta, base_seed=BASE_SEED)
#         clf.fit(bundle.DL, bundle.y)   # Loss-Kurve aus clf._loss_history ziehen
print("Modelle bereit:", GlobalOld.__name__, GlobalNew.__name__, LabelOld.__name__, LabelNew.__name__)